# Prepare Quick-Test Assets from Already-Saved Files

This notebook **does not rerun data preprocessing or model training**.

It uses the files that the full experiment already saved:

```text
datasets/test_data.pt
51-NHITS_model.pth        # preferred
# or seq2seq_model.pth
```

It creates:

```text
quick_test_assets/
├── quick_test_data.pt
├── model_state_dict.pth
├── manifest.json
└── target_scaler.joblib   # created only if recoverable
```

The quick-test subset uses a few **complete held-out test graphs**. Spatial nodes are not cropped, because keeping the complete graph preserves the exact graph-neighborhood context used by the trained GNN.

If the original target scaler was not saved, this notebook will try to recover its affine inverse transformation from a previously saved `51results_well_*_my_model.npz` file. If recovery is not possible, the public quick-test notebook still runs and reports normalized RMSE/MAE plus R²/NSE.

In [ ]:
from pathlib import Path
import json
import shutil
import numpy as np
import torch
import joblib

from sklearn.preprocessing import RobustScaler

ROOT = Path(".")
TEST_FILE = ROOT / "datasets" / "test_data.pt"

MODEL_CANDIDATES = [
    ROOT / "51-NHITS_model.pth",
    ROOT / "seq2seq_model.pth",
]

ASSET_DIR = ROOT / "quick_test_assets"
ASSET_DIR.mkdir(parents=True, exist_ok=True)

QUICK_DATA_FILE = ASSET_DIR / "quick_test_data.pt"
MODEL_OUT_FILE = ASSET_DIR / "model_state_dict.pth"
SCALER_OUT_FILE = ASSET_DIR / "target_scaler.joblib"
MANIFEST_FILE = ASSET_DIR / "manifest.json"

# Two or three test times are enough for a rapid verification.
N_TEMPORAL_SAMPLES = 3

print("Working directory:", ROOT.resolve())
print("Expected test file:", TEST_FILE)

## 1. Load the already-saved held-out test set

In [ ]:
if not TEST_FILE.exists():
    raise FileNotFoundError(
        f"Cannot find {TEST_FILE}. "
        "Point TEST_FILE to the test_data.pt that your earlier experiment already saved."
    )

try:
    test_data = torch.load(TEST_FILE, map_location="cpu", weights_only=False)
except TypeError:
    # Compatibility with older PyTorch versions.
    test_data = torch.load(TEST_FILE, map_location="cpu")

if not isinstance(test_data, (list, tuple)) or len(test_data) == 0:
    raise RuntimeError("test_data.pt is empty or is not a list/tuple of graph samples.")

print(f"Loaded {len(test_data)} held-out test graph samples.")
g0 = test_data[0]
print("First graph:")
print("  nodes:", g0.num_nodes)
print("  edges:", g0.edge_index.shape[1])
print("  y:", tuple(g0.y.shape))
print("  x_static:", tuple(g0.x_static.shape))
print("  x_dynamic_encoder:", tuple(g0.x_dynamic_encoder.shape))

## 2. Select a few complete test graphs

In [ ]:
n_take = min(N_TEMPORAL_SAMPLES, len(test_data))
positions = np.unique(
    np.linspace(0, len(test_data) - 1, n_take).round().astype(int)
).tolist()

quick_test_data = []
sample_manifest = []

for pos in positions:
    g = test_data[pos].cpu()
    # Keep the complete graph. This avoids changing neighborhood information.
    quick_test_data.append(g)

    sample_manifest.append({
        "source_test_position": int(pos),
        "t": int(g.t) if hasattr(g, "t") else None,
        "date": str(g.date) if hasattr(g, "date") else None,
        "num_nodes": int(g.num_nodes),
        "num_edges": int(g.edge_index.shape[1]),
        "forecast_horizon": int(g.y.shape[1]),
    })

torch.save(quick_test_data, QUICK_DATA_FILE)

print("Selected test positions:", positions)
print("Saved:", QUICK_DATA_FILE)
print("Number of quick-test graphs:", len(quick_test_data))

## 3. Copy the already-trained model checkpoint

In [ ]:
model_source = None
for candidate in MODEL_CANDIDATES:
    if candidate.exists():
        model_source = candidate
        break

if model_source is None:
    raise FileNotFoundError(
        "Could not find a trained checkpoint. Expected one of:\n  - "
        + "\n  - ".join(str(p) for p in MODEL_CANDIDATES)
    )

shutil.copy2(model_source, MODEL_OUT_FILE)

print("Model checkpoint source:", model_source)
print("Copied to:", MODEL_OUT_FILE)

## 4. Recover the target scaler if possible

Your earlier `save_datasets(...)` cell saved the graph datasets, but it **did not save `processor.target_scaler`**.

This cell therefore tries, in order:

1. an existing `target_scaler.joblib`;
2. a `processed_data.pt` created by `HydroDataProcessor.save_processed_data(...)`;
3. a saved `51results_well_*_my_model.npz`.

The third option works because your saved NPZ contains groundwater values already converted back to real units, while `test_data.pt` contains the corresponding normalized targets. Since `RobustScaler.inverse_transform` is affine, the scale and centre can be recovered from these paired values.

If none is available, the quick test can still verify inference and report R²/NSE and normalized RMSE/MAE.

In [ ]:
scaler_status = {
    "available": False,
    "source": None,
    "recovery_max_abs_error": None,
}

# ---- Option A: already saved scaler ----
existing_scaler_candidates = [
    ROOT / "target_scaler.joblib",
    ROOT / "datasets" / "target_scaler.joblib",
]

for p in existing_scaler_candidates:
    if p.exists():
        shutil.copy2(p, SCALER_OUT_FILE)
        scaler_status.update({
            "available": True,
            "source": str(p),
        })
        print("Found existing target scaler:", p)
        break

# ---- Option B: processed_data.pt metadata ----
if not scaler_status["available"]:
    processed_candidates = [
        ROOT / "processed_data.pt",
        ROOT / "datasets" / "processed_data.pt",
    ]

    for p in processed_candidates:
        if not p.exists():
            continue
        try:
            try:
                obj = torch.load(p, map_location="cpu", weights_only=False)
            except TypeError:
                obj = torch.load(p, map_location="cpu")

            scaler = obj.get("metadata", {}).get("target_scaler", None) if isinstance(obj, dict) else None
            if scaler is not None:
                joblib.dump(scaler, SCALER_OUT_FILE)
                scaler_status.update({
                    "available": True,
                    "source": str(p) + " -> metadata.target_scaler",
                })
                print("Recovered target scaler from:", p)
                break
        except Exception as e:
            print(f"Could not read scaler from {p}: {e}")

# ---- Option C: recover affine inverse transform from saved real-value NPZ ----
if not scaler_status["available"]:
    npz_candidates = sorted(ROOT.glob("51results_well_*_my_model.npz"))

    for npz_path in npz_candidates:
        try:
            saved = np.load(npz_path, allow_pickle=True)

            required_keys = {"node_idx", "test_true"}
            if not required_keys.issubset(saved.files):
                continue

            node_idx = int(np.asarray(saved["node_idx"]).item())
            real_true = np.asarray(saved["test_true"], dtype=float).reshape(-1)

            n = min(len(test_data), len(real_true))
            if n < 3:
                continue

            norm_true = np.array([
                float(test_data[i].y[node_idx, 0].detach().cpu().item())
                for i in range(n)
            ], dtype=float)

            real_true = real_true[:n]

            mask = np.isfinite(norm_true) & np.isfinite(real_true)
            if mask.sum() < 3 or np.std(norm_true[mask]) < 1e-12:
                continue

            # inverse transform has the form: real = normalized * scale + center
            A = np.column_stack([norm_true[mask], np.ones(mask.sum())])
            scale, center = np.linalg.lstsq(A, real_true[mask], rcond=None)[0]

            reconstructed = norm_true[mask] * scale + center
            max_abs_err = float(np.max(np.abs(reconstructed - real_true[mask])))

            # A genuine match should be extremely close. Allow a small tolerance for
            # floating-point roundoff and values saved through NumPy.
            tolerance = 1e-3

            print(
                f"Scaler recovery candidate {npz_path.name}: "
                f"scale={scale:.10g}, center={center:.10g}, "
                f"max_abs_error={max_abs_err:.6g}"
            )

            if scale > 0 and max_abs_err <= tolerance:
                recovered_scaler = RobustScaler()
                recovered_scaler.center_ = np.array([center], dtype=float)
                recovered_scaler.scale_ = np.array([scale], dtype=float)
                recovered_scaler.n_features_in_ = 1

                joblib.dump(recovered_scaler, SCALER_OUT_FILE)

                scaler_status.update({
                    "available": True,
                    "source": str(npz_path) + " -> affine recovery",
                    "recovery_max_abs_error": max_abs_err,
                })

                print("Successfully reconstructed target scaler from:", npz_path)
                break

        except Exception as e:
            print(f"Could not use {npz_path.name}: {e}")

if scaler_status["available"]:
    print("\nPhysical-unit RMSE/MAE will be available.")
    print("Scaler saved to:", SCALER_OUT_FILE)
else:
    print("\nNo target scaler could be recovered.")
    print("This is NOT fatal.")
    print("The public quick test will report:")
    print("  - normalized RMSE")
    print("  - normalized MAE")
    print("  - R²")
    print("  - NSE")
    print("R² and NSE are unchanged by an affine target scaling.")

## 5. Save the manifest

In [ ]:
manifest = {
    "description": (
        "Quick-test subset extracted from the already-saved held-out experimental test set. "
        "Complete graph samples are retained so the GNN neighborhood context is unchanged."
    ),
    "source_test_file": str(TEST_FILE),
    "source_model_file": str(model_source),
    "selected_test_positions": positions,
    "samples": sample_manifest,
    "target_scaler": scaler_status,
}

with open(MANIFEST_FILE, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

print("=" * 72)
print("QUICK-TEST ASSET PREPARATION COMPLETED")
print("=" * 72)
print("Created:")
print(" -", QUICK_DATA_FILE)
print(" -", MODEL_OUT_FILE)
print(" -", MANIFEST_FILE)
if scaler_status["available"]:
    print(" -", SCALER_OUT_FILE)
else:
    print(" - target_scaler.joblib: not available (optional in v3)")